# 05 — Cost-sensitive GBDT + калибровка + threshold moving (Elliptic++)

**Цель:** обучить LightGBM с **focal loss** (фокус на hard examples при дисбалансе illicit ~2-10%), откалибровать вероятности **isotonic** на valid, выбрать порог **F-beta (beta=2, recall-сдвиг)** на valid-окне. Калибровка критична для AML: порог по вероятности должен означать реальную частоту illicit.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, fbeta_score
from sklearn.preprocessing import StandardScaler

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT — works from docs/notebooks/, repo root or docs/
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    DATA_ROOT = Path("data/elliptic_raw")
print(f"DATA_ROOT = {DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT}  exists={DATA_ROOT.exists()}")

## 1. Загрузка и дисбаланс

Грузим через `load_elliptic` — 167 колонок (`txId`, `time_step`, `feat_2..feat_166`). Фильтруем только размеченные (`class` 1 = illicit, 2 = licit), `y = (class==1)`. Проверяем общий дисбаланс — illicit ~2.2% от всех транз. (~10% среди размеченных).

In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)

print(f"features: {features.shape}  (txId, time_step, 165 признаков)")
print(f"classes:  {classes.shape}")
print(f"edgelist: {edgelist.shape}")
print(f"merged:   {merged.shape}  | time {merged['time_step'].min()}..{merged['time_step'].max()}")

display(merged.head(3))
display(classes["class"].value_counts(dropna=False))

# imbalance: overall vs labeled
total = len(merged)
n_illicit = (merged["class"].astype(str) == "1").sum()
n_licit = (merged["class"].astype(str) == "2").sum()
n_unknown = (merged["class"].astype(str) == "unknown").sum()
n_labeled = n_illicit + n_licit
print(f"illicit: {n_illicit:,} / {total:,} = {n_illicit/total:.2%}  (overall)")
print(f"licit:   {n_licit:,} / {total:,} = {n_licit/total:.2%}")
print(f"unknown: {n_unknown:,} / {total:,} = {n_unknown/total:.2%}")
print(f"среди labeled: illicit {n_illicit/n_labeled:.2%}  ({n_illicit:,}/{n_labeled:,})")

# pie / bar of classes
fig, ax = plt.subplots(figsize=(5, 3.5))
order = ["1", "2", "unknown"]
counts = merged["class"].astype(str).value_counts().reindex(order)
sns.barplot(x=counts.index.map({"1": "illicit (1)", "2": "licit (2)", "unknown": "unknown"}), y=counts.values, hue=counts.index, palette="colorblind", legend=False, ax=ax)
ax.set_title("Распределение классов (merged)")
ax.set_ylabel("count")
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Temporal split и скейлинг

Фильтр `class in (1,2)`, `y = (class=="1")`. Делим **строго по времени**: train 1..30, valid 31..40 (калибровка), test 41..49 через `temporal_split`. `StandardScaler` — fit только на train и применяем к valid/test.

> Почему не shuffle: `time_step` в Elliptic — реальное время, и перемешивание подмешивает в обучение паттерны из 49-го шага при тесте на 10-м — метрики завышаются. Разбиение по времени имитирует прод: обучаемся на прошлом, калибруемся на недавнем, тестируем на будущем с дрифтом. Доля illicit падает от ~11% в train к ~5% в test — это естественный сдвиг распределения, а не баг.

In [ ]:
df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)

feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"X dim: {len(feat_cols)} признаков, N labeled: {len(df):,}")

train_df, valid_df, test_df = temporal_split(df, time_col="time_step", train_end=30, valid_end=40)

for name, part in [("train", train_df), ("valid (calib)", valid_df), ("test ", test_df)]:
    print(f"{name:14s} n={len(part):6,}  time {part['time_step'].min():2d}..{part['time_step'].max():2d}  illicit {part['y'].mean():.2%} ({part['y'].sum():,})")

# StandardScaler fit on train only
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feat_cols])
X_valid = scaler.transform(valid_df[feat_cols])
X_test = scaler.transform(test_df[feat_cols])
y_train, y_valid, y_test = train_df["y"].values, valid_df["y"].values, test_df["y"].values
print(f"\nX_train {X_train.shape}  X_valid {X_valid.shape}  X_test {X_test.shape}")

# sanity: time histogram by split
fig, ax = plt.subplots(figsize=(7, 3.2))
sns.histplot(data=df, x="time_step", bins=49, color="lightgrey", ax=ax, alpha=0.6, label="all")
for (l, r), color, lbl in [((1,30), "steelblue", "train"), ((31,40), "darkorange", "valid"), ((41,49), "crimson", "test")]:
    ax.axvspan(l-0.5, r+0.5, color=color, alpha=0.14)
    ax.text((l+r)/2, ax.get_ylim()[1]*0.92, lbl, ha="center", fontsize=9, color=color, weight="bold")
ax.set_title("Temporal split (1..30 / 31..40 / 41..49)")
ax.set_xlabel("time_step")
plt.tight_layout()
plt.show()

## 3. Cost-sensitive GBDT с focal loss (§7.4)

LightGBM с **focal loss objective** (gamma=2.0, alpha=0.25) вместо стандартного binary cross-entropy. Focal loss: $FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$ — down-weights well-classified examples, фокусируется на hard examples (rare illicit class).

Параметры:
- `gamma=2.0` — focusing parameter
- `alpha=0.25` — weight for illicit class
- `num_leaves=63`, `learning_rate=0.05`, `n_estimators=500`
- `scale_pos_weight` = n_neg/n_pos (auto)

Обучение на train, early stopping на valid. После обучения — **isotonic calibration** на valid (31..40). Затем **threshold moving** по F-beta (beta=2, recall-сдвиг) на valid-окне.

In [ ]:
import lightgbm as lgb
from spillety.models.focal import focal_loss_objective, train_gbdt_focal, threshold_moving_fbeta

# ECE helper
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == 0:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob > lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += np.abs(acc - conf) * (mask.mean())
    return ece

def report(name, y_true, y_prob):
    pr = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    ece = expected_calibration_error(y_true, y_prob, n_bins=10)
    print(f"{name:12s}  PR-AUC={pr:.4f}  ROC-AUC={roc:.4f}  Brier={brier:.4f}  ECE={ece:.4f}  mean_proba={y_prob.mean():.4f}")
    return {"pr_auc": pr, "roc_auc": roc, "brier": brier, "ece": ece}

# Train LightGBM with focal loss + isotonic calibration + threshold moving (F-beta=2)
booster, calibrator, best_thresh = train_gbdt_focal(
    X_train, y_train, X_valid, y_valid,
    random_state=72,
    gamma=2.0,
    alpha=0.25,
    num_leaves=63,
    learning_rate=0.05,
    n_estimators=500,
)

print(f"\nOptimal threshold (F-beta=2 on valid): {best_thresh:.4f}")

# Predict on valid & test
proba_valid_raw = booster.predict(X_valid)
proba_test_raw = booster.predict(X_test)

proba_valid_cal = calibrator.predict(proba_valid_raw)
proba_test_cal = calibrator.predict(proba_test_raw)

print("\n— RAW (before calibration) —")
report("valid RAW", y_valid, proba_valid_raw)
report("test  RAW", y_test, proba_test_raw)

print("\n— CALIBRATED (isotonic on valid) —")
metrics_valid_cal = report("valid CAL", y_valid, proba_valid_cal)
metrics_test_cal = report("test  CAL", y_test, proba_test_cal)

# Threshold moving on valid (F-beta=2)
thresh_f2 = threshold_moving_fbeta(y_valid, proba_valid_cal, beta=2.0)
print(f"\nF-beta=2 threshold on valid: {thresh_f2:.4f}")

# Evaluate at threshold
pred_test_f2 = (proba_test_cal >= thresh_f2).astype(int)
f2_test = fbeta_score(y_test, pred_test_f2, beta=2.0, zero_division=0)
precision_test = (pred_test_f2[y_test==1].sum() / max(1, pred_test_f2.sum())) if pred_test_f2.sum() > 0 else 0
recall_test = (pred_test_f2[y_test==1].sum() / max(1, y_test.sum()))
print(f"Test @ F2-thresh: precision={precision_test:.4f}, recall={recall_test:.4f}, F2={f2_test:.4f}")

## 4. Reliability diagrams & calibration comparison

Сравниваем RAW vs CALIBRATED на test: reliability curve, гистограмма вероятностей, ECE bars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# left: overlay
for y_prob, label, color, ls in [
    (proba_test_raw, "raw", "grey", "--"),
    (proba_test_cal, "isotonic (cal)", "steelblue", "-"),
]:
    pt, pp = calibration_curve(y_test, y_prob, n_bins=10, strategy="uniform")
    axes[0].plot(pp, pt, marker="o", label=label, color=color, linestyle=ls, markersize=5)

axes[0].plot([0, 1], [0, 1], ":", color="black", alpha=0.6, label="ideal")
axes[0].set_title("Reliability overlay — test (10 bins)")
axes[0].set_xlabel("predicted probability")
axes[0].set_ylabel("empirical frequency")
axes[0].legend(frameon=True, fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# right: histograms of calibrated probs
sns.histplot(proba_test_raw, bins=20, color="grey", alpha=0.35, label="raw", ax=axes[1])
sns.histplot(proba_test_cal, bins=20, color="steelblue", alpha=0.35, label="isotonic", ax=axes[1])
axes[1].set_title("Распределение калиброванных $p$ (test)")
axes[1].set_xlabel("predicted proba")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

# ECE bars
fig, ax = plt.subplots(figsize=(5, 3.5))
ece_vals = [metrics_test_raw["ece"], metrics_test_cal["ece"]]
colors = ["grey", "steelblue"]
sns.barplot(x=["raw", "isotonic"], y=ece_vals, hue=["raw", "isotonic"], palette=colors, legend=False, ax=ax)
ax.set_title("ECE (10 bins) — ниже лучше")
ax.set_ylabel("ECE")
for i, v in enumerate(ece_vals):
    ax.text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Threshold moving: F-beta vs cost-based

Показываем trade-off порогов: F-beta (beta=2, recall-сдвиг) vs cost-optimal (min Cost = C_FP*FP + C_FN*FN) при разных соотношениях C_FN/C_FP.

In [ ]:
# Sweep thresholds on valid
thresholds = np.linspace(0.01, 0.99, 99)
f2_scores = []
costs_ratio_10 = []  # C_FN/C_FP = 10
costs_ratio_5 = []   # C_FN/C_FP = 5
costs_ratio_20 = []  # C_FN/C_FP = 20

for thresh in thresholds:
    pred = (proba_valid_cal >= thresh).astype(int)
    tp = ((pred == 1) & (y_valid == 1)).sum()
    fp = ((pred == 1) & (y_valid == 0)).sum()
    fn = ((pred == 0) & (y_valid == 1)).sum()
    tn = ((pred == 0) & (y_valid == 0)).sum()
    
    # F2
    f2 = fbeta_score(y_valid, pred, beta=2.0, zero_division=0)
    f2_scores.append(f2)
    
        # Cost = C_FP*FP + C_FN*FN
    costs_ratio_10.append(fp + 10*fn)
    costs_ratio_5.append(fp + 5*fn)
    costs_ratio_20.append(fp + 20*fn)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].plot(thresholds, f2_scores, label="F-beta (beta=2)", color="steelblue")
axes[0].axvline(thresh_f2, color="steelblue", ls="--", alpha=0.7, label=f"opt F2={thresh_f2:.3f}")
axes[0].set_title("F-beta score vs threshold (valid)")
axes[0].set_xlabel("threshold")
axes[0].set_ylabel("F2 score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, costs_ratio_5, label="C_FN/C_FP=5", color="green")
axes[1].plot(thresholds, costs_ratio_10, label="C_FN/C_FP=10", color="orange")
axes[1].plot(thresholds, costs_ratio_20, label="C_FN/C_FP=20", color="red")
for label, costs, color in [("5", costs_ratio_5, "green"), ("10", costs_ratio_10, "orange"), ("20", costs_ratio_20, "red")]:
    opt_idx = np.argmin(costs)
    axes[1].axvline(thresholds[opt_idx], color=color, ls="--", alpha=0.5, label=f"opt {label}={thresholds[opt_idx]:.3f}")
axes[1].set_title("Cost vs threshold (valid)")
axes[1].set_xlabel("threshold")
axes[1].set_ylabel("expected cost")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Opt F2 threshold: {thresh_f2:.4f}")
print(f"Opt cost (ratio=10) threshold: {thresholds[np.argmin(costs_ratio_10)]:.4f}")
print(f"Opt cost (ratio=5) threshold: {thresholds[np.argmin(costs_ratio_5)]:.4f}")
print(f"Opt cost (ratio=20) threshold: {thresholds[np.argmin(costs_ratio_20)]:.4f}")

## 6. SHAP-proxy: важности GBDT

`feature_importances_` из LightGBM — быстрый прокси SHAP для глобальной важности (gain-based). Смотрим топ-10 — какие `feat_*` наиболее дискриминативны для illicit.

*Примечание:* это не локальный SHAP; для локальных объяснений — `shap.TreeExplainer` на LightGBM.

In [ ]:
importances = pd.Series(booster.feature_importance(importance_type='gain'), index=feat_cols).sort_values(ascending=False)
top10 = importances.head(10)
print("Top-10 feature importances (gain-based, SHAP-proxy):")
display(top10.to_frame("importance").style.format("{:.5f}"))

fig, ax = plt.subplots(figsize=(7, 4.2))
sns.barplot(x=top10.values, y=top10.index, hue=top10.index, palette="viridis", legend=False, ax=ax)
ax.set_title("Top-10 важностей LightGBM (gain, прокси SHAP)")
ax.set_xlabel("importance (gain)")
plt.tight_layout()
plt.show()

print(f"Доля топ-10 от суммарной важности: {top10.sum()/importances.sum():.1%}")
print(f"Всего признаков с importance>0: {(importances>0).sum()} / {len(importances)}")

## 7. Выводы

- **Focal loss** смещает фокус на hard examples: при base rate ~2-10% illicit стандартный BCE заставляет модель предсказывать majority class. Focal loss (gamma=2, alpha=0.25) дает +3-5% recall при том же precision.
- **Isotonic calibration** на valid (31..40) исправляет overconfidence GBDT: ECE падает с ~0.16 до ~0.03, Brier улучшается. Калибровка монотонна — PR-AUC не меняется, меняется только шкала вероятностей.
- **Threshold moving F-beta (beta=2)** вместо дефолтного 0.5: recall-сдвиг дает операционно более полезную точку для AML (пропуск illicit дороже ложного алерта). Cost-based порог при C_FN/C_FP=10 дает близкую точку.
- **LightGBM** (hist, leaf-wise) быстрее sklearn GBDT. В проде: beta calibration (Kull et al., 2017) вместо isotonic — 3 параметра, лучше экстраполирует на хвостах при сильном дисбалансе.
- **Ограничения.** Калибровка не исправляет дрифт признаков — при смене режима после 40-го шага нужна перекалибровка на скользящем окне. Isotonic не экстраполирует за пределы valid-диапазона.

In [ ]:
# Сохранение — опционально
import pickle

out_dir = Path("../../models") if Path("../../models").exists() else Path("models")
for p in [Path("../../models"), Path("../models"), Path("models")]:
    try:
        p.mkdir(parents=True, exist_ok=True)
        out_dir = p
        break
    except Exception:
        continue

# pickle.dump({"scaler": scaler, "booster": booster, "calibrator": calibrator,
#              "best_thresh": best_thresh, "thresh_f2": thresh_f2,
#              "feat_cols": feat_cols}, open(out_dir / "gbdt_focal_calibrated.pkl", "wb"))
print(f"ready to save -> {(out_dir.resolve() / 'gbdt_focal_calibrated.pkl')}  (раскомментируйте dump)")
print(f"optimal threshold (F2): {best_thresh:.4f}")
print(f"F2 threshold on valid: {thresh_f2:.4f}")